## Assignment: Building a Mixture of Experts (MoE) Router

In [1]:
%pip install python-dotenv --upgrade --quiet langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**Load API Key and Initialize Base Model**

In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

MODEL_NAME = "llama-3.1-8b-instant"

Enter your Groq API Key:  ········


**Define Expert Configurations (Mixture of Experts)**

In [3]:
MODEL_CONFIG = {
    "technical": "You are a senior technical support engineer. Provide precise, logical, and code-focused solutions.",
    
    "billing": "You are a billing support specialist. Be empathetic and clearly explain charges, refunds, or payment policies.",
    
    "general": "You are a friendly assistant helping with general questions."
}

**Router Function — Classifies User Query into Expert Category**

In [4]:
def route_prompt(user_input):
    
    router_llm = ChatGroq(
        model=MODEL_NAME,
        temperature=0.0   
    )
    
    prompt = f"""
Classify the following user query into one of these categories:
technical, billing, general.

Return ONLY the category name.

Query: {user_input}
"""
    
    response = router_llm.invoke(prompt)
    
    category = response.content.strip().lower()
    return category

**Tool Use**

In [5]:
def get_bitcoin_price():
    return "The current Bitcoin price is approximately $65,000 (mock data)."

**Orchestrator — Routes Request to Correct Expert and Generates Response**

In [6]:
def process_request(user_input):
    
    category = route_prompt(user_input)
    
    if "bitcoin" in user_input.lower():
        return get_bitcoin_price()
    
    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])
    
    expert_llm = ChatGroq(
        model=MODEL_NAME,
        temperature=0.7
    )
    
    final_prompt = f"""
{system_prompt}

User Query: {user_input}
"""
    
    response = expert_llm.invoke(final_prompt)
    
    return response.content

**Test the Mixture of Experts Router with Sample Queries**

In [7]:
queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hello, how are you?",
    "What is the current price of Bitcoin?"
]

for q in queries:
    print("User:", q)
    print("Category:", route_prompt(q))
    print("Response:", process_request(q))
    print("-" * 50)

User: My python script is throwing an IndexError on line 5.
Category: technical
Response: **Troubleshooting IndexError in Python Script**

When encountering an `IndexError` in Python, it typically occurs when trying to access an element in a sequence (such as a list, tuple, or string) with an index that is out of range. This can happen when the index is greater than or equal to the length of the sequence.

**Example Code**

```python
# Example Python script
my_list = [1, 2, 3]
print(my_list[3])  # Throws IndexError: list index out of range
```

**Solution Steps**

1. **Check the length of the sequence**: Verify if the index being used is within the valid range of the sequence.
2. **Verify the index calculation**: Ensure that the index is calculated correctly and does not exceed the sequence's length.
3. **Provide a valid alternative**: If the index is out of range, provide a valid alternative index or use an alternative method to access the element.

**Code-Focused Solution**

To resol